# Build End-to-End Pipeline Dataset


Orders data with:
1. NULL values
2. String salary (amount as string)
3. Date column

---

## Tasks

1. Clean NULLs
2. Cast columns
3. Add derived columns
4. Apply UDF
5. Full Load
6. Incremental Load
7. Parameterize notebook

---

## 1. SAMPLE DATASET (Dirty + Realistic)

- NULLs in amount
- String salary
- Dates
- Updates (for incremental load)



from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    # Do not use this for first time: (3, "C003", "Tablet", "22000", "2024-01-06"), # updated
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]
df = spark.createDataFrame(data, columns)


---

## TASK LIST

### Task 1: Handle NULLs
- Replace NULL amount with 1000

### Task 2: Cast Columns
- Convert amount → integer
- Convert updated_date → date

### Task 3: Add Derived Columns
- bonus = amount * 0.1
- category:
  - amount >= 20000 → High
  - else → Low

### Task 4: Apply UDF
- Create amount_bucket:
  - < 10000 → Low
  - 10000-30000 → Medium
  - > 30000 → High

### Task 5: Full Load
- Load all data to target

### Task 6: Incremental Load
- Load only new/updated records
- Handle duplicates (keep latest)

### Task 7: Parameterization
- Pass input path, output path, last loaded date, load type as parameters


In [0]:
# Create sample dataset with NULLs, string salary, and dates

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

# Initial data (for first full load)
data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)

print("Initial Dataset:")
df.show()
df.printSchema()

Initial Dataset:
+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  NULL|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  NULL|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- updated_date: string (nullable = true)



In [0]:
# Replace NULL amount with 1000

df_clean = df.fillna({"amount": "1000"})

print("After handling NULLs:")
df_clean.show()

After handling NULLs:
+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  1000|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  1000|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+



In [0]:
# Convert amount to integer and updated_date to date


df_clean = df_clean.withColumn("amount", col("amount").cast(IntegerType())) \
                   .withColumn("updated_date", to_date(col("updated_date")))

print("After casting columns:")
df_clean.show()
df_clean.printSchema()

After casting columns:
+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  1000|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  1000|  2024-01-05|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- updated_date: date (nullable = true)



In [0]:
# Add bonus column (amount * 0.1) and category column

df_clean = df_clean.withColumn("bonus", col("amount") * 0.1)

df_clean = df_clean.withColumn(
    "category",
    when(col("amount") >= 20000, "High").otherwise("Low")
)

print("After adding derived columns:")
df_clean.show()

After adding derived columns:
+--------+-----------+----------+------+------------+------+--------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|
+--------+-----------+----------+------+------------+------+--------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|
|       2|       C002|    Mobile|  1000|  2024-01-02| 100.0|     Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|    High|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|
|       5|       C005|Headphones|  1000|  2024-01-05| 100.0|     Low|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     Low|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|     Low|
+--------+-----------+----------+------+------------+------+--------+



In [0]:
# Create UDF to categorize amount into buckets
#Create amount_bucket:
#< 10000 → Low
#10000–30000 → Medium
#30000 → High
def categorize_amount(amount):
    if amount<10000:
        return 'Low'
    elif amount>=10000 and amount<=30000:
        return 'Meduim'
    else:
        return 'High'
categorize_udf=udf(categorize_amount,StringType())
df_clean=df_clean.withColumn("amount_bucket",categorize_udf(col("amount")))
print("After applying UDF:")
df_clean.display()


After applying UDF:


order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Meduim
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,Meduim
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Meduim
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


In [0]:
# Full load: Write all data to target table

target_path = "/Volumes/workspace/default/day1/new/"

# Use coalesce(1) to write as a single file instead of multiple part files
df_clean.coalesce(1).write.mode("overwrite").parquet(target_path)

print(f"Full load completed to {target_path}")
print(f"Total records: {df_clean.count()}")

# Verify the load
df_loaded = spark.read.parquet(target_path)
print("\nLoaded data:")
df_loaded.show()

Full load completed to /Volumes/workspace/default/day1/new/
Total records: 8

Loaded data:
+--------+-----------+----------+------+------------+------+--------+-------------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|amount_bucket|
+--------+-----------+----------+------+------------+------+--------+-------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|         High|
|       2|       C002|    Mobile|  1000|  2024-01-02| 100.0|     Low|          Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|    High|       Meduim|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|         High|
|       5|       C005|Headphones|  1000|  2024-01-05| 100.0|     Low|          Low|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|       Meduim|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     Low|       Meduim|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|     Low|

In [0]:
# Simulate incremental data (includes updated record for order_id 3)

incremental_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06"),  # Updated
    (9, "C009", "Speaker", "12000", "2024-01-08"),  # New
    (10, "C010", "Monitor", "45000", "2024-01-09")  # New
]

df_incremental = spark.createDataFrame(incremental_data, columns)

print("Incremental data:")
df_incremental.show()

# Apply same transformations
df_incremental_clean = df_incremental.fillna({"amount": "1000"}) \
    .withColumn("amount", col("amount").cast(IntegerType())) \
    .withColumn("updated_date", to_date(col("updated_date"))) \
    .withColumn("bonus", col("amount") * 0.1) \
    .withColumn("category", when(col("amount") >= 20000, "High").otherwise("Low")) \
    .withColumn("amount_bucket", categorize_udf(col("amount")))

print("\nTransformed incremental data:")
df_incremental_clean.show()

# Load existing data
df_existing = spark.read.parquet(target_path)

# Merge: union and keep latest by order_id (deduplication)
from pyspark.sql.window import Window

df_merged = df_existing.union(df_incremental_clean)

# Keep the record with the latest updated_date for each order_id
window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())
df_merged = df_merged.withColumn("rank", row_number().over(window_spec)) \
                     .filter(col("rank") == 1) \
                     .drop("rank")

print("\nAfter incremental load (merged data):")
df_merged.orderBy("order_id").show()

# Save back
df_merged.write.mode("overwrite").parquet(target_path)
print(f"Incremental load completed. Total records: {df_merged.count()}")

Incremental data:
+--------+-----------+-------+------+------------+
|order_id|customer_id|product|amount|updated_date|
+--------+-----------+-------+------+------------+
|       3|       C003| Tablet| 22000|  2024-01-06|
|       9|       C009|Speaker| 12000|  2024-01-08|
|      10|       C010|Monitor| 45000|  2024-01-09|
+--------+-----------+-------+------+------------+


Transformed incremental data:
+--------+-----------+-------+------+------------+------+--------+-------------+
|order_id|customer_id|product|amount|updated_date| bonus|category|amount_bucket|
+--------+-----------+-------+------+------------+------+--------+-------------+
|       3|       C003| Tablet| 22000|  2024-01-06|2200.0|    High|       Meduim|
|       9|       C009|Speaker| 12000|  2024-01-08|1200.0|     Low|       Meduim|
|      10|       C010|Monitor| 45000|  2024-01-09|4500.0|    High|         High|
+--------+-----------+-------+------+------------+------+--------+-------------+


After incremental load (

In [0]:
# Parameterize the notebook for flexible execution

from datetime import datetime

# Create widgets for parameters
dbutils.widgets.text("input_path", "/tmp/orders_source", "Input Path")
dbutils.widgets.text("output_path", "/tmp/orders_target", "Output Path")
dbutils.widgets.text("last_loaded_date", "2024-01-07", "Last Loaded Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"], "Load Type")

# Read parameters
input_path = dbutils.widgets.get("input_path")
output_path = dbutils.widgets.get("output_path")
last_loaded_date = dbutils.widgets.get("last_loaded_date")
load_type = dbutils.widgets.get("load_type")

print("Pipeline Parameters:")
print(f"Input Path: {input_path}")
print(f"Output Path: {output_path}")
print(f"Last Loaded Date: {last_loaded_date}")
print(f"Load Type: {load_type}")

Pipeline Parameters:
Input Path: /tmp/orders_source
Output Path: /tmp/orders_target
Last Loaded Date: 2024-01-07
Load Type: incremental


In [0]:
# Execute pipeline based on parameters

def transform_data(df):
    """Apply all transformations to a dataframe"""
    return df.fillna({"amount": "1000"}) \
        .withColumn("amount", col("amount").cast(IntegerType())) \
        .withColumn("updated_date", to_date(col("updated_date"))) \
        .withColumn("bonus", col("amount") * 0.1) \
        .withColumn("category", when(col("amount") >= 20000, "High").otherwise("Low")) \
        .withColumn("amount_bucket", categorize_udf(col("amount")))

# Read parameters
load_type = dbutils.widgets.get("load_type")
last_loaded_date = dbutils.widgets.get("last_loaded_date")

if load_type == "full":
    print("Executing FULL LOAD...")
    df_source = df  # Use original data
    df_transformed = transform_data(df_source)
    df_transformed.write.mode("overwrite").parquet(target_path)
    print(f"Full load completed: {df_transformed.count()} records")
    
elif load_type == "incremental":
    print(f"Executing INCREMENTAL LOAD (after {last_loaded_date})...")
    
    # Filter only new/updated records
    df_source = df_incremental  # Use incremental data
    df_new = df_source.filter(col("updated_date") > last_loaded_date)
    
    if df_new.count() == 0:
        print("No new records to load")
    else:
        df_transformed = transform_data(df_new)
        
        # Read existing and merge
        df_existing = spark.read.parquet(target_path)
        df_merged = df_existing.union(df_transformed)
        
        # Deduplication
        window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())
        df_final = df_merged.withColumn("rank", row_number().over(window_spec)) \
                           .filter(col("rank") == 1) \
                           .drop("rank")
        
        df_final.write.mode("overwrite").parquet(target_path)
        print(f"Incremental load completed: {df_new.count()} new records processed")
        print(f"Total records in target: {df_final.count()}")

print("\nFinal Data:")
spark.read.parquet(target_path).orderBy("order_id").show()

In [0]:
# Generate summary report

df_final = spark.read.parquet(target_path)

print("="*70)
print("END-TO-END PIPELINE SUMMARY")
print("="*70)

print(f"\nTotal Records: {df_final.count()}")

print("\nRecords by Category:")
df_final.groupBy("category").count().show()

print("Records by Amount Bucket:")
df_final.groupBy("amount_bucket").count().show()

print("Total Amount and Bonus:")
df_final.agg(
    sum("amount").alias("total_amount"),
    sum("bonus").alias("total_bonus"),
    avg("amount").alias("avg_amount")
).show()

print("Top 5 Orders by Amount:")
df_final.orderBy(col("amount").desc()).select("order_id", "customer_id", "product", "amount", "category").show(5)

print("\n" + "="*70)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("="*70)